# Practice 4: Bounded loop (attempt, check, retry, give up)

**The idea:** a step runs, a check looks at the result, and if it is not good enough the graph goes around again. The danger is a loop that never ends. The rule: **keep the counter in the state, and let a pure router stop the loop when the counter reaches the limit.**

**The flow:** `attempt` adds 10 to the score and 1 to the try count. The router then picks one of three exits:
1. score is 50 or more: go to `passed`
2. otherwise, if tries have reached `MAX_TRIES` (3): go to `human_review`, which pauses for a yes or no
3. otherwise: go round again to `attempt`

Passing is checked *before* the limit, so passing on the very last allowed try still counts as a pass.

**What changed from the first version**
1. **The tests could not run.** The first test file never imported `build_graph`, `judge`, `InMemorySaver` or `Command`, because a test file runs on its own and does not see notebook cells. It also had `parameterize` (it is `parametrize`), and the `graph` fixture had no `return`. Code now lives in `bounded_loop.py`, and the tests import from it.
2. **Several expected values were wrong.** The first message is `Attempt 1` text built from the *old* count (0), a score of 200 never reaches the human step, and one test was copied from the expense exercise ("director") and made no sense here.
3. **`recursion_limit` was in the wrong place.** It must sit at the top level of the config, not inside `"configurable"`. Inside, it is silently ignored, so the "unbounded" demo never showed an error.
4. **The worst-case test was missing.** The exercise's "done when" is a check that never passes, ending in the escalation node after exactly `MAX_TRIES` tries, proven by a test.
5. **Smaller:** the router has an explicit map of names, magic numbers became named constants, the reply `"YES"` is now case and space tolerant, thread ids are strings, and nodes are renamed (`attempt`, `passed`, `human_review`) because `start_flow` was really the loop body, not a start.

## Step 1. The module (logic only)

In [1]:
%%writefile bounded_loop.py
from typing import TypedDict, Literal, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt

MAX_TRIES = 3      # the bound: no more than this many attempts
PASS_SCORE = 50    # a score at or above this passes
STEP = 10          # stand-in for "the model improves a bit each time"


class State(TypedDict):
    try_count: int
    score: int
    messages: Annotated[list[str], operator.add]


# ---- nodes: take the state, return only the changes ----
def attempt(state: State) -> dict:
    n = state["try_count"] + 1
    score = state["score"] + STEP
    return {
        "try_count": n,
        "score": score,
        "messages": [f"Attempt {n}: score {score}"],
    }


def passed(state: State) -> dict:
    return {"messages": [f"Passed on attempt {state['try_count']}"]}


def human_review(state: State) -> dict:
    answer = interrupt({"question": f"Gave up after {state['try_count']} attempts. Approve? yes or no"})
    approved = str(answer).strip().lower() == "yes"
    return {"messages": [f"Manual verdict: {'approved' if approved else 'not approved'}"]}


# ---- the router: a PURE function. Pass wins; then the bound; else go round again ----
def route_after_attempt(state: State) -> Literal["passed", "attempt", "human_review"]:
    if state["score"] >= PASS_SCORE:
        return "passed"
    if state["try_count"] >= MAX_TRIES:
        return "human_review"
    return "attempt"


def build_graph(checkpointer=None, router=route_after_attempt):
    """`router` can be swapped, so we can build a broken, unbounded graph on purpose."""
    builder = StateGraph(State)
    builder.add_node("attempt", attempt)
    builder.add_node("passed", passed)
    builder.add_node("human_review", human_review)

    builder.add_edge(START, "attempt")
    builder.add_conditional_edges(
        "attempt",
        router,
        {"attempt": "attempt", "passed": "passed", "human_review": "human_review"},
    )
    builder.add_edge("passed", END)
    builder.add_edge("human_review", END)
    return builder.compile(checkpointer=checkpointer)

Writing bounded_loop.py


## Step 2. Try the router by hand
It is a pure function, so you can test it with plain dicts: no graph, no pause. Look at the boundary rows.

In [2]:
from bounded_loop import route_after_attempt, build_graph, MAX_TRIES

print(f"{'score':>5} {'tries':>5}  ->  route")
for score, tries in [(49, 1), (50, 1), (49, 2), (49, 3), (50, 3), (0, 8)]:
    r = route_after_attempt({"score": score, "try_count": tries, "messages": []})
    print(f"{score:>5} {tries:>5}  ->  {r}")

score tries  ->  route
   49     1  ->  attempt
   50     1  ->  passed
   49     2  ->  attempt
   49     3  ->  human_review
   50     3  ->  passed
    0     8  ->  human_review


## Step 3. Look at the graph
The arrow from `attempt` back to `attempt` is the loop.

In [3]:
print(build_graph().get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	attempt(attempt)
	passed(passed)
	human_review(human_review)
	__end__([<p>__end__</p>]):::last
	__start__ --> attempt;
	attempt -.-> human_review;
	attempt -.-> passed;
	human_review --> __end__;
	passed --> __end__;
	attempt -.-> attempt;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 4. Run four cases, each on a fresh thread

In [4]:
import uuid
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

graph = build_graph(InMemorySaver())
new_config = lambda: {"configurable": {"thread_id": str(uuid.uuid4())}}
start = lambda score: {"try_count": 0, "score": score, "messages": []}

for label, score in [("passes on try 1", 40), ("passes on try 2", 30),
                     ("passes on the LAST try", 20), ("never passes", -1000)]:
    res = graph.invoke(start(score), new_config())
    paused = "  <-- PAUSED for a human" if "__interrupt__" in res else ""
    print(f"{label:<24} tries={res['try_count']}  last={res['messages'][-1]!r}{paused}")

passes on try 1          tries=1  last='Passed on attempt 1'
passes on try 2          tries=2  last='Passed on attempt 2'
passes on the LAST try   tries=3  last='Passed on attempt 3'
never passes             tries=3  last='Attempt 3: score -970'  <-- PAUSED for a human


## Step 5. The worst case, and how to look inside a paused run

`get_state(config)` tells you where a thread is. `.next` names the node it is waiting at, and `.values` is the saved state. This is how the test proves "it stopped at human review after exactly 3 tries".

In [5]:
cfg = new_config()
graph.invoke(start(-1000), cfg)
snap = graph.get_state(cfg)
print("waiting at :", snap.next)
print("try_count  :", snap.values["try_count"])
print("messages   :", snap.values["messages"])

print()
print("resume with ' YES ' ->", graph.invoke(Command(resume=" YES "), cfg)["messages"][-1])

waiting at : ('human_review',)
try_count  : 3
messages   : ['Attempt 1: score -990', 'Attempt 2: score -980', 'Attempt 3: score -970']

resume with ' YES ' -> Manual verdict: approved


## Step 6. Break it on purpose: no bound, and the misplaced limit

The exercise says to delete the bound and watch the error. Here the router always answers "go round again". LangGraph's `recursion_limit` is the seat belt that stops it. Here we set a small limit ourselves; the default in this version is about 10,000, far too loose to rely on. Then the second half shows the mistake from the first version: the limit placed inside `configurable` does nothing.

In [6]:
from langgraph.errors import GraphRecursionError

broken = build_graph(InMemorySaver(), router=lambda state: "attempt")
try:
    broken.invoke(start(0), {"configurable": {"thread_id": "b"}, "recursion_limit": 10})
except GraphRecursionError as e:
    print("unbounded:", type(e).__name__, "-", str(e).splitlines()[0])

print()
wrong = {"configurable": {"thread_id": "w1", "recursion_limit": 2}}      # limit ignored
print("limit inside configurable -> ran fine, tries =", graph.invoke(start(10), wrong)["try_count"])

right = {"configurable": {"thread_id": "w2"}, "recursion_limit": 2}      # limit applied
try:
    graph.invoke(start(10), right)
except GraphRecursionError as e:
    print("limit at top level        ->", type(e).__name__)

unbounded: GraphRecursionError - Recursion limit of 10 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.

limit inside configurable -> ran fine, tries = 3
limit at top level        -> GraphRecursionError


## Step 7. The tests

New ideas compared with the router exercise:

- **`graph.get_state(config).next`** to prove which node a paused run is waiting at.
- **Boundary rows in `parametrize`**: score 49 vs 50, and tries `MAX_TRIES - 1` vs `MAX_TRIES`. Also the row `(50, MAX_TRIES)` which checks that passing on the last allowed try still passes.
- **The worst-case test:** a check that never passes ends in `human_review` after exactly `MAX_TRIES` attempts.
- **A bound test over many starting scores:** the attempt count never exceeds the limit, with no `GraphRecursionError` at the default limit.
- **`pytest.raises(GraphRecursionError)`** for the deliberately broken, unbounded graph.
- **A test that documents a stumble:** the misplaced `recursion_limit`.

In [7]:
%%writefile test_bounded_loop.py
import copy
import uuid

import pytest
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphRecursionError
from langgraph.types import Command

from bounded_loop import MAX_TRIES, build_graph, route_after_attempt


# ---------------------------------------------------------------- fixtures
@pytest.fixture
def graph():
    """A fresh graph with its own memory for every test."""
    return build_graph(InMemorySaver())      # note the return!


@pytest.fixture
def config():
    """A brand-new thread id for every test."""
    return {"configurable": {"thread_id": str(uuid.uuid4())}}


def start(score):
    return {"try_count": 0, "score": score, "messages": []}


def visited(graph, config, score):
    """Names of the nodes that ran, in order."""
    names = []
    for chunk in graph.stream(start(score), config, stream_mode="updates"):
        names.extend(k for k in chunk if k != "__interrupt__")
    return names


# ------------------------------------------------- 1. the router (no graph)
@pytest.mark.parametrize(
    "score, try_count, expected",
    [
        (49, 1, "attempt"),         # just below the pass mark
        (50, 1, "passed"),          # exactly the pass mark
        (49, MAX_TRIES - 1, "attempt"),      # last attempt that still allows another
        (49, MAX_TRIES, "human_review"),     # the bound is reached
        (50, MAX_TRIES, "passed"),           # passing on the last allowed try still passes
        (0, MAX_TRIES + 5, "human_review"),  # past the bound still stops
    ],
)
def test_router_picks_the_right_branch(score, try_count, expected):
    state = {"score": score, "try_count": try_count, "messages": []}
    assert route_after_attempt(state) == expected


def test_router_does_not_change_the_state():
    state = {"score": 20, "try_count": 1, "messages": ["earlier"]}
    before = copy.deepcopy(state)
    route_after_attempt(state)
    assert state == before


# --------------------------------- 2. the three cases the exercise asks for
def test_passes_on_the_second_try(graph, config):
    result = graph.invoke(start(30), config)          # 40, then 50
    assert "__interrupt__" not in result
    assert result["try_count"] == 2
    assert result["messages"] == ["Attempt 1: score 40", "Attempt 2: score 50", "Passed on attempt 2"]


def test_passes_exactly_on_the_last_allowed_try(graph, config):
    result = graph.invoke(start(20), config)          # 30, 40, then 50 on try 3
    assert "__interrupt__" not in result
    assert result["try_count"] == MAX_TRIES
    assert result["messages"][-1] == f"Passed on attempt {MAX_TRIES}"


def test_never_passes_ends_in_human_review_after_exactly_max_tries(graph, config):
    """THE worst-case test: a check that never passes must not loop forever."""
    graph.invoke(start(-1000), config)
    snapshot = graph.get_state(config)
    assert snapshot.next == ("human_review",)                 # paused at the escalation node
    assert snapshot.values["try_count"] == MAX_TRIES
    attempts = [m for m in snapshot.values["messages"] if m.startswith("Attempt")]
    assert len(attempts) == MAX_TRIES


def test_path_taken_is_exactly_what_we_expect(graph, config):
    assert visited(graph, config, 30) == ["attempt", "attempt", "passed"]


# -------------------------------------------- 3. the human step: pause, resume
@pytest.mark.parametrize("answer, verdict", [("yes", "approved"), (" YES ", "approved"), ("no", "not approved"), ("maybe", "not approved")])
def test_human_verdict(graph, config, answer, verdict):
    paused = graph.invoke(start(-1000), config)
    assert "Gave up after 3 attempts" in paused["__interrupt__"][0].value["question"]
    final = graph.invoke(Command(resume=answer), config)
    assert final["messages"][-1] == f"Manual verdict: {verdict}"


# ------------------------------------------------ 4. the bound really holds
@pytest.mark.parametrize("score", [-1000, -50, 0, 10, 19, 20, 30, 40, 49, 50, 90])
def test_attempts_never_exceed_the_bound(graph, score):
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    result = graph.invoke(start(score), config)
    assert result["try_count"] <= MAX_TRIES


# ------------------------------------- 5. break it on purpose: no bound at all
def always_again(state):
    return "attempt"


def test_without_a_bound_only_the_recursion_limit_saves_us():
    broken = build_graph(InMemorySaver(), router=always_again)
    config = {"configurable": {"thread_id": "broken"}, "recursion_limit": 10}   # top level!
    with pytest.raises(GraphRecursionError):
        broken.invoke(start(0), config)


def test_recursion_limit_inside_configurable_is_silently_ignored(graph):
    """Documents a stumble: put recursion_limit at the top level of the config, not inside 'configurable'."""
    wrong = {"configurable": {"thread_id": "w", "recursion_limit": 2}}
    result = graph.invoke(start(10), wrong)                     # needs 4 steps, limit ignored: runs fine
    assert result["try_count"] == MAX_TRIES

    right = {"configurable": {"thread_id": "r"}, "recursion_limit": 2}
    with pytest.raises(GraphRecursionError):
        graph.invoke(start(10), right)

Writing test_bounded_loop.py


## Step 8. Run the tests

In [8]:
!python -m pytest -v --no-header -p no:cacheprovider

============================= test session starts ==============================
collecting ... 

collected 28 items                                                             

test_bounded_loop.py::test_router_picks_the_right_branch[49-1-attempt] PASSED [  3%]
test_bounded_loop.py::test_router_picks_the_right_branch[50-1-passed] PASSED [  7%]
test_bounded_loop.py::test_router_picks_the_right_branch[49-2-attempt] PASSED [ 10%]
test_bounded_loop.py::test_router_picks_the_right_branch[49-3-human_review] PASSED [ 14%]
test_bounded_loop.py::test_router_picks_the_right_branch[50-3-passed] PASSED [ 17%]
test_bounded_loop.py::test_router_picks_the_right_branch[0-8-human_review] PASSED [ 21%]
test_bounded_loop.py::test_router_does_not_change_the_state PASSED       [ 25%]
test_bounded_loop.py::test_passes_on_the_second_try PASSED               [ 28%]
test_bounded_loop.py::test_passes_exactly_on_the_last_allowed_try PASSED [ 32%]
test_bounded_loop.py::test_never_passes_ends_in_human_review_after_exactly_max_tries PASSED [ 35%]


test_bounded_loop.py::test_path_taken_is_exactly_what_we_expect PASSED   [ 39%]
test_bounded_loop.py::test_human_verdict[yes-approved] PASSED            [ 42%]
test_bounded_loop.py::test_human_verdict[ YES -approved] PASSED          [ 46%]
test_bounded_loop.py::test_human_verdict[no-not approved] PASSED         [ 50%]
test_bounded_loop.py::test_human_verdict[maybe-not approved] PASSED      [ 53%]
test_bounded_loop.py::test_attempts_never_exceed_the_bound[-1000] PASSED [ 57%]
test_bounded_loop.py::test_attempts_never_exceed_the_bound[-50] PASSED   [ 60%]
test_bounded_loop.py::test_attempts_never_exceed_the_bound[0] 

PASSED     [ 64%]
test_bounded_loop.py::test_attempts_never_exceed_the_bound[10] PASSED    [ 67%]
test_bounded_loop.py::test_attempts_never_exceed_the_bound[19] PASSED    [ 71%]
test_bounded_loop.py::test_attempts_never_exceed_the_bound[20] PASSED    [ 75%]
test_bounded_loop.py::test_attempts_never_exceed_the_bound[30] PASSED    [ 78%]
test_bounded_loop.py::test_attempts_never_exceed_the_bound[40] PASSED    [ 82%]
test_bounded_loop.py::test_attempts_never_exceed_the_bound[49] PASSED    [ 85%]
test_bounded_loop.py::test_attempts_never_exceed_the_bound[50] PASSED    [ 89%]
test_bounded_loop.py::test_attempts_never_exceed_the_bound[90] PASSED    [ 92%]
test_bounded_loop.py::test_without_a_bound_only_the_recursion_limit_saves_us 

PASSED [ 96%]
test_bounded_loop.py::test_recursion_limit_inside_configurable_is_silently_ignored PASSED [100%]

============================== 28 passed in 0.37s ==============================


## Step 9. Check that the tests can actually fail

Two broken routers: one uses `>` instead of `>=` (an off-by-one in the bound), and one has no bound at all. The numbers show what the tests would catch.

Notice that the no-bound version did **not** raise an error. In this LangGraph version the default `recursion_limit` is about 10,000 steps, so the built-in seat belt only trips after a huge number of steps. If each step called a paid model, that would be a lot of calls. The real bound has to be in your state and router, and a test has to prove it.

In [9]:
def off_by_one(state):                 # uses > instead of >=
    if state["score"] >= 50: return "passed"
    return "human_review" if state["try_count"] > MAX_TRIES else "attempt"

g = build_graph(InMemorySaver(), router=off_by_one)
res = g.invoke(start(-1000), new_config())
print(f"off-by-one: ran {res['try_count']} attempts (limit is {MAX_TRIES})  -> the 'exactly MAX_TRIES' test would fail")

def no_bound(state):                   # forgot the limit entirely
    return "passed" if state["score"] >= 50 else "attempt"

g = build_graph(InMemorySaver(), router=no_bound)
res = g.invoke(start(-1000), new_config())
print(f"no bound  : ran {res['try_count']} attempts (limit is {MAX_TRIES})  -> the 'bound holds' test would fail")

off-by-one: ran 4 attempts (limit is 3)  -> the 'exactly MAX_TRIES' test would fail


no bound  : ran 105 attempts (limit is 3)  -> the 'bound holds' test would fail


## Your turn

1. Restart the kernel and run all cells. Everything should pass.
2. Change the router so that a score of 90 or more skips the loop entirely and goes straight to `passed`, even if `try_count` is 0. Write the failing test first.
3. Add a test that if a human answers "no", the final message mentions "not approved" and `try_count` is still `MAX_TRIES`.
4. Add to `stumbles.txt` anything you hesitated on: "where does recursion_limit go?", "which check comes first, pass or bound?", "what does get_state().next tell me?"